# 📱 SMS Spam Classification using Machine Learning (SVM & Ensemble Benchmarking)

**Dataset:** [Kaggle / UCI SMS Spam Collection Dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

## 📌 Project Overview
This notebook presents an end-to-end Machine Learning pipeline for classifying SMS messages into **Ham** (legitimate) or **Spam**. It incorporates:
- Automated dataset loading & duplicate removal
- Advanced text cleaning (regex cleaning, stopword removal, WordNet lemmatization)
- Exploratory Data Analysis (EDA) with visualization & WordClouds
- Feature extraction via TF-IDF Vectorization (N-grams & Sublinear Scaling)
- Multi-model benchmarking (Multinomial Naive Bayes, Linear SVC, Logistic Regression, Random Forest)
- Hyperparameter tuning via 5-Fold Stratified `GridSearchCV`
- Model evaluation (Accuracy, Precision, Recall, F1-Score, Confusion Matrix, ROC-AUC)
- Pipeline serialization using `joblib` for production deployment.

## 🛠️ 1. Dependencies & Environment Setup

In [ ]:
# Install dependencies if running in Colab or fresh environment
# !pip -q install wordcloud nltk scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import os
import re
import string
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve
)

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Styling settings for graphics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
print("[*] All essential packages successfully imported!")

## 📂 2. Dataset Ingestion & Inspection

In [ ]:
# Automatic dataset download/loading
data_path = "spam.csv"
if not os.path.exists(data_path) and os.path.exists("SMS.csv"):
    data_path = "SMS.csv"
elif not os.path.exists(data_path):
    from download_data import download_dataset
    data_path = download_dataset()

try:
    df = pd.read_csv(data_path, encoding="utf-8")
except Exception:
    df = pd.read_csv(data_path, encoding="latin-1")

# Standardize column names
if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']].copy()
    df.columns = ['label', 'message']
else:
    df = df[['label', 'message']].copy()

print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
print("--- Dataset Summary ---")
print(df.info())
print("\nMissing values:", df.isnull().sum().to_dict())
print("Duplicate rows:", df.duplicated(subset=['message']).sum())
print("\nClass Distribution:")
print(df['label'].value_counts())

## 📊 3. Exploratory Data Analysis (EDA)

In [ ]:
# Compute character length and word counts
df['char_count'] = df['message'].str.len()
df['word_count'] = df['message'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class Balance Bar Chart
sns.countplot(data=df, x='label', ax=axes[0], palette=['#4C72B0', '#DD8452'])
axes[0].set_title("SMS Class Distribution (Ham vs Spam)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")

# Word Count Distribution
sns.histplot(data=df, x='word_count', hue='label', bins=35, kde=True, ax=axes[1], palette=['#4C72B0', '#DD8452'])
axes[1].set_title("Word Count Distribution per Category", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# WordCloud Visualizations for Spam and Ham
spam_words = ' '.join(df[df['label'].astype(str).str.lower() == 'spam']['message'])
ham_words = ' '.join(df[df['label'].astype(str).str.lower() == 'ham']['message'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wc_spam = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(spam_words)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title("Frequent Words in SPAM Messages", fontsize=14, fontweight='bold', color='crimson')

wc_ham = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(ham_words)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title("Frequent Words in HAM Messages", fontsize=14, fontweight='bold', color='darkgreen')

plt.tight_layout()
plt.show()

## 🧹 4. Data Preprocessing & Cleaning

In [ ]:
stop_words = set(stopwords.words('english'))
lemm = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)  # Remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)       # Remove emails
    text = re.sub(r'\d+', ' ', text)           # Remove digits
    text = text.translate(str.maketrans('', '', string.punctuation)) # Remove punctuation
    tokens = [
        lemm.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]
    return ' '.join(tokens)

# Remove duplicates
df = df.drop_duplicates(subset=['message']).reset_index(drop=True)

# Preprocess text
df['clean_message'] = df['message'].apply(clean_text)

# Encode labels: ham -> 0, spam -> 1
df['label'] = df['label'].astype(str).str.strip().str.lower().map({'ham': 0, 'spam': 1})
df = df.dropna(subset=['label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f"Processed dataset size: {len(df)}")
df[['message', 'clean_message', 'label']].head()

## ✂️ 5. Train/Test Stratified Split

In [ ]:
X = df['clean_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape} (Ham: {sum(y_train==0)}, Spam: {sum(y_train==1)})")
print(f"Testing set shape : {X_test.shape} (Ham: {sum(y_test==0)}, Spam: {sum(y_test==1)})")

## ⚔️ 6. Multi-Model Benchmarking

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVC': LinearSVC(C=1.0, random_state=42, max_iter=2000),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    if hasattr(pipe.named_steps['clf'], 'predict_proba'):
        probs = pipe.predict_proba(X_test)[:, 1]
        auc_val = roc_auc_score(y_test, probs)
    elif hasattr(pipe.named_steps['clf'], 'decision_function'):
        probs = pipe.decision_function(X_test)
        auc_val = roc_auc_score(y_test, probs)
    else:
        auc_val = np.nan
        
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc_val
    })

benchmark_df = pd.DataFrame(results)
benchmark_df.sort_values(by='F1-Score', ascending=False, inplace=True)
benchmark_df.style.highlight_max(axis=0, color='lightgreen')

## ⚙️ 7. Hyperparameter Tuning (GridSearchCV)

In [ ]:
tune_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', CalibratedClassifierCV(estimator=LinearSVC(random_state=42, max_iter=2000)))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 8000],
    'tfidf__min_df': [1, 2],
    'clf__estimator__C': [0.1, 0.5, 1.0, 2.0]
}

grid = GridSearchCV(tune_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("[*] Best Hyperparameters Found:", grid.best_params_)
print(f"[*] Best Cross-Validation F1-Score: {grid.best_score_:.4f}")

## 📈 8. Comprehensive Model Evaluation

In [ ]:
test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)[:, 1]

print("--- Final Classification Report ---")
print(classification_report(y_test, test_preds, target_names=['Ham', 'Spam']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title("Confusion Matrix", fontsize=12, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_probs)
auc_score = roc_auc_score(y_test, test_probs)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=12, fontweight='bold')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 💾 9. Model Serialization

In [ ]:
# Export model pipeline using joblib
model_filename = "sms_spam_pipeline.pkl"
joblib.dump(best_model, model_filename)
print(f"[+] Model successfully serialized and saved to '{model_filename}'!")

## 🔮 10. Real-World Inference Function

In [ ]:
def predict_sms(text):
    cleaned_input = clean_text(text)
    prediction = best_model.predict([cleaned_input])[0]
    proba = best_model.predict_proba([cleaned_input])[0][prediction]
    label = "SPAM 🚨" if prediction == 1 else "HAM ✅"
    print(f"Input Message : '{text}'")
    print(f"Prediction    : {label} (Confidence: {proba*100:.2f}%)\n")

# Test sample messages
predict_sms("FreeMsg: Claim your £1000 cash prize now! Text CLAIM to 81010 to get your reward instantly!")
predict_sms("Hey Mom, I'll be home late tonight. Please don't wait up for dinner.")
predict_sms("URGENT! Your banking password has expired. Click http://bit.ly/fake-bank to renew immediately.")